# Method Choice

This project uses a Random Forest Classifier.

The goal is to predict whether a page is likely to experience declining search performance so that content teams can prioritize refresh efforts.

Random Forest was selected because:

- It can model non-linear relationships between search signals.
- It handles interactions between multiple features automatically.
- It is more robust than a single Decision Tree.
- It provides feature importance scores that improve interpretability.

The baseline from Week 4 uses manually written scoring rules. This model attempts to learn patterns automatically from historical search data while being evaluated on the same prediction task.

# Validation Design

The data is split using GroupShuffleSplit based on client_hash_id.

This ensures that pages from the same client do not appear in both the training and testing sets.

This validation strategy better reflects how the model would generalize to unseen clients and reduces the risk of overly optimistic performance estimates.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

In [ ]:
# Data already created in previous notebooks

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=feature_cols)

X = model_data[feature_cols]

y = model_data["is_declining"]

groups = model_data["client_hash_id"]

In [ ]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(
        X,
        y,
        groups
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [ ]:
model = RandomForestClassifier(

    n_estimators=200,

    random_state=42,

    n_jobs=-1

)

model.fit(

    X_train,

    y_train

)

In [ ]:
predictions = model.predict(X_test)

probabilities = model.predict_proba(X_test)[:,1]

In [ ]:
print(classification_report(

    y_test,

    predictions,

    digits=3

))

In [ ]:
accuracy = accuracy_score(

    y_test,

    predictions

)

precision = precision_score(

    y_test,

    predictions

)

recall = recall_score(

    y_test,

    predictions

)

f1 = f1_score(

    y_test,

    predictions

)

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

# Comparison with Week 4 Baseline

The Week 4 baseline used manually defined thresholds and simple scoring rules.

The Random Forest model uses the same features but learns interactions automatically.

The comparison below evaluates both approaches on the same prediction task.

In [ ]:
baseline_prediction = (

    X_test["imp_prev30"] <

    X_test["imp_prev30"].median()

).astype(int)

baseline_precision = precision_score(

    y_test,

    baseline_prediction

)

model_precision = precision_score(

    y_test,

    predictions

)

comparison = pd.DataFrame({

    "Model":[

        "Week 4 Baseline",

        "Random Forest"

    ],

    "Precision":[

        baseline_precision,

        model_precision

    ]

})

comparison

In [ ]:
importance = pd.DataFrame({

    "Feature":feature_cols,

    "Importance":model.feature_importances_

})

importance.sort_values(

    "Importance",

    ascending=False

)

# Error Analysis

Some pages predicted as declining may actually remain stable.

Conversely, some declining pages may not be detected by the model.

Possible explanations include:

- seasonal traffic variation
- incomplete historical information
- external search trends
- content updates not captured by the available features

The model should therefore support human decision-making rather than replace editorial judgment.

# Interpretation

The Random Forest model captures relationships between multiple search signals more effectively than the manually designed baseline.

Feature importance suggests that previous impressions and query-related metrics contribute strongly to predicting declining pages.

Although the model improves prediction performance, it should be interpreted as identifying statistical patterns rather than causal relationships.